# Practical Class 1: End-to-End Regression Masterclass

Welcome to the first of our three final practical classes! Today, we are bridging the gap between theoretical math and production-ready code. 

We will be tackling the **Ames Housing Dataset**, the absolute gold standard for classical regression. Your goal is to predict the `SalePrice` of a house given 79 different features. This dataset is notoriously messy: it has missing values, extreme outliers, and highly correlated features.

In this notebook, you will build a robust Machine Learning pipeline, compare fundamentally different algorithm families, and finally combine them to achieve a Kaggle-winning score. 

*Note: Ensure you have downloaded `train.csv` and `test.csv` from the [Kaggle House Prices competition](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) and placed them in the same folder as this notebook.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# You will need to import your sklearn modules here as you go!

sns.set_theme(style="whitegrid")

# Load the data
try:
    df = pd.read_csv('train.csv')
    print(f"Training Data loaded successfully! Shape: {df.shape}")
except FileNotFoundError:
    print("Error: Please download train.csv from Kaggle and put it in this directory.")

## Step 1: The Target Variable Trap

Before we look at our 79 features, we must look at what we are trying to predict: `SalePrice`.

**Your Task:**
1. Plot a histogram of `SalePrice`. What do you notice about its shape?
2. If you train a model on this skewed target, the MSE will heavily penalize mistakes on multi-million dollar mansions and ignore normal houses. Fix this by creating a new column called `LogSalePrice` that normalizes this distribution.

**Hints:**
* *Use `sns.histplot()` to visualize the distribution.*
* *To fix the skew, apply a log transformation. The safest function to use is `np.log1p()` (Log(1 + x)), which handles zeros safely.*
* *Remember: when you eventually make predictions, you will need to reverse this using `np.expm1()`!*

In [ ]:
# YOUR CODE HERE

## Step 2: The Production-Ready Pipeline

In the real world, you cannot manually fill NaNs and encode variables line-by-line. It causes **Data Leakage** during cross-validation and makes deploying your model impossible. We must build a `Pipeline`.

**Your Task:**
1. Split your features (`X`) and your transformed target (`y`). Drop the 'Id' column.
2. Identify which columns are numerical and which are categorical.
3. Build a `ColumnTransformer` that applies:
    * To Numerical features: Impute missing values with the median, then apply Standard Scaling.
    * To Categorical features: Impute missing values with the mode (most frequent), then apply One-Hot Encoding (ignore unknown categories).

**Hints:**
* *To grab column names by type, look into `X.select_dtypes(include=...)`.*
* *You will need `Pipeline`, `ColumnTransformer`, `SimpleImputer`, `StandardScaler`, and `OneHotEncoder` from `sklearn.pipeline` and `sklearn.preprocessing`.*
* *For the categorical OneHotEncoder, set `handle_unknown='ignore'` so it doesn't crash on unseen test data.*

In [ ]:
# YOUR CODE HERE

## Step 3: Linear vs. Trees (The Face-off)

Now that our data is flowing safely through a pipeline, let's train two fundamentally different models and compare their Cross-Validation scores.

**Your Task:**
1. Create a full pipeline that combines your `ColumnTransformer` from Step 2 with a **Ridge Regression** (L2 Regularization) model.
2. Create a second full pipeline that combines the transformer with a **Random Forest** or **XGBoost** model.
3. Evaluate both using 5-Fold Cross-Validation. Print their average Root Mean Squared Error (RMSE).

**Hints:**
* *To combine your preprocessor and a model, use `Pipeline(steps=[('preprocessor', your_ct), ('model', Ridge())])`.*
* *Use `cross_val_score` from `sklearn.model_selection`.*
* *Scikit-learn's scoring parameter expects utility (higher is better), so use `scoring='neg_root_mean_squared_error'`. Remember to multiply the result by -1 to make it positive!*

In [ ]:
# YOUR CODE HERE

## Step 4: The Sandbox (Explore & Improve)

This is your time to experiment. The models above were just baselines. How can you lower your CV error?

**Your Task (Choose one or more):**
1. **Hyperparameter Tuning:** Use `GridSearchCV` to find the optimal `alpha` for your Ridge model, or the optimal `max_depth` for your Tree model.
2. **Algorithm Swapping:** Try `Lasso` regression. What happens to the coefficients? Try `KNeighborsRegressor` or `SVR`.
3. **Feature Expansion:** Insert `PolynomialFeatures(degree=2)` into your numerical pipeline. Does allowing features to multiply together help the linear model?

*Write your code below and see how low you can push your RMSE!*

In [ ]:
# YOUR CODE HERE

## Step 5: The Kaggle Secret (Ensembling)

Tree models (like XGBoost) make blocky, step-based predictions. Linear models (like Ridge) make smooth, geometric predictions. Because they make *different types of mistakes*, combining them almost always beats using them individually.

**Your Task:**
1. Fit your best Linear Pipeline on the full dataset (`X`, `y`).
2. Fit your best Tree Pipeline on the full dataset (`X`, `y`).
3. Predict on the dataset with both models.
4. Create a final training prediction array by simply averaging the two prediction arrays: `(linear_preds + tree_preds) / 2`.
5. (Optional but highly recommended): Calculate the MSE of this combined prediction to prove to yourself that the ensemble beats the individual models.

In [ ]:
# YOUR CODE HERE

## Step 6: The Kaggle Submission (Mini-Competition!)

It is time to test your model against the world. We are going to make live predictions and submit them to Kaggle. Whoever gets the lowest RMSE on the public leaderboard by the end of the class wins!

**Your Task:**
1. Load `test.csv`.
2. Notice that `test.csv` **does not have** a `SalePrice` column. 
3. Pass the `test.csv` features through your fitted ensemble pipeline from Step 5 to generate your predictions.
4. **CRITICAL:** Your model just predicted `LogSalePrice`. You must convert these predictions back to normal dollar amounts using `np.expm1()` before submitting, or your score will be terrible!
5. Create a pandas DataFrame with exactly two columns: `Id` (from the test set) and `SalePrice` (your un-logged predictions).
6. Save this DataFrame to a CSV file (e.g., `submission.csv`) without the index.
7. Go to the Kaggle competition page and click **Submit Predictions**!

**Hints:**
* *To save to CSV without the pandas index numbers: `submission_df.to_csv('submission.csv', index=False)`*

In [ ]:
# 1. Load test.csv

# 2. Extract the features (make sure to drop 'Id' if you did that during training!)

# 3. Predict using your ensemble or best pipeline

# 4. Reverse the log transformation using np.expm1()

# 5. Create the submission DataFrame

# 6. Export to CSV